In [1]:
from manim import *
import jupyter_capture_output

video_scene = " -v WARNING --disable_caching tunnel_compression_Scene"
image_scene = f" -v WARNING --disable_caching -r {2*427},{2*240}  -s tunnel_compression_Scene"

Jupyter Capture Output v0.0.11


In [342]:
# parameters
l = 10				# tunnel length
width = 1			# width of the pressure pulse
v_t = 1.0			# train velocity
v_s = 3.0			# speed of sound
t_0 = 1.5			# time the pulse starts from the train etering the tunnel


# range of the left-running wave
def get_range_left(t_raw):
	# get the modulo of the time
	t = t_raw % (2*l / v_t)
	# x-center of the left-running wave
	wave_center = l - v_t*(t+t_0)
	# leftwards propagation only
	if t < (l-width/2) / v_t - t_0:
		return [wave_center-width/2, wave_center+width/2]
	# superposition on left
	elif (l-width/2) / v_t - t_0 < t and t < (l+width/2) / v_t - t_0:
		return [0, wave_center+width/2]
	# rightwards propagation only
	elif (l+width/2) / v_t - t_0 < t and t < (3*l-width/2) / v_t - t_0:
		return [0, 0]
	# superposition on right
	elif (3*l-width/2) / v_t - t_0 < t and t < (3*l+width/2) / v_t - t_0:
		return [v_t*(t+t_0) -3*l + width/2, 10]
	else:
		return ValueError
	

# range of the right-running wave
def get_range_right(t_raw):
	# get the modulo of the time
	t = t_raw % (2*l / v_t)
	# x-center of the left-running wave
	wave_center = v_t*(t+t_0) - l
	# leftwards propagation only
	if t < (l-width/2) / v_t - t_0:
		return [0, 0]
	# superposition on left
	elif (l-width/2) / v_t - t_0 < t and t < (l+width/2) / v_t - t_0:
		return [0, wave_center+width/2]
	# rightwards propagation only
	elif (l+width/2) / v_t - t_0 < t and t < (3*l-width/2) / v_t - t_0:
		return [wave_center-width/2, wave_center+width/2]
	# superposition on right
	elif (3*l-width/2) / v_t - t_0 < t and t < (3*l+width/2) / v_t - t_0:
		return [wave_center-width/2, 10]
	else:
		return ValueError


# baseline pressure
def baseline(x):
	return 0


# wave running to the left
def get_pressure_left(t):
	def pressure_left(x):
		p_left = -np.sin(x/width*2*PI-v_t*(t+t_0) + l)
		return p_left
	return pressure_left


# wave running to the right
def get_pressure_right(t):
	def pressure_right(x):
		p_right = np.sin(x/width*2*PI+v_t*(t+t_0) - l)
		return p_right
	return pressure_right



def get_interval_pressure_left(t):
	def interval_pressure_left(x):
		x_mod = x % (2*l)
		# x-center of the left-running wave
		wave_center = (l - v_t*(t+t_0)) % (2*l)
		border_left = wave_center-width/2
		border_right = wave_center+width/2
		if (border_left < x_mod and x_mod < border_right):
			return -np.sin(x/width*2*PI-v_t*(t+t_0) + l)
		else:
			return 0
	return interval_pressure_left


def get_interval_pressure_right(t):
	def interval_pressure_right(x):
		x_mod = x % (2*l)
		# x-center of the left-running wave
		wave_center = v_t*(t+t_0) - l
		border_left = wave_center-width/2
		border_right = wave_center+width/2
		if (border_left < x_mod and x_mod < border_right):
			return np.sin(x/width*2*PI+v_t*(t+t_0) - l)
		else:
			return 0
	return interval_pressure_right


def get_pressure_pulse(t):
	def pressure_pulse(x):
		x_mod = x % (2*l)
		p = 0
		# rightwards wave
		right_wave_center = (v_t*t_0 + v_s*t) % (2*l)
		right_wave_border_left = right_wave_center-width/2
		right_wave_border_right = right_wave_center+width/2
		if (right_wave_border_left < x_mod and x_mod < right_wave_border_right):
			p += np.sin((x - v_t*t_0 - v_s*t) / width*2*PI)

		# leftwards wave
		left_wave_center = (2*l - v_t*t_0 - v_s*t) % (2*l)
		left_wave_border_left = left_wave_center-width/2
		left_wave_border_right = left_wave_center+width/2
		if (left_wave_border_left < x_mod and x_mod < left_wave_border_right):
			p += np.sin((x + v_t*t_0 + v_s*t - 2*l) / width*2*PI)
		return p
	return pressure_pulse

In [343]:
class Tunnel(Mobject):
	def __init__(self, tunnel_center = np.array([0, 0, 0]), tunnel_height = 1, tunnel_width = 5, **kwargs):
		super().__init__(**kwargs)
        
		self.tunnel_center = tunnel_center
		self.tunnel_height = tunnel_height
		self.tunnel_width = tunnel_width
        
		print(self.tunnel_width, tunnel_width)
		upper_tunnel = Line(start = self.tunnel_center-self.tunnel_width/2*RIGHT+self.tunnel_height/2*UP, end = self.tunnel_center+self.tunnel_width/2*RIGHT+self.tunnel_height/2*UP, stroke_width = 4, stroke_color = BLACK)
		lower_tunnel = Line(start = self.tunnel_center-self.tunnel_width/2*RIGHT-self.tunnel_height/2*UP, end = self.tunnel_center+self.tunnel_width/2*RIGHT-self.tunnel_height/2*UP, stroke_width = 4, stroke_color = BLACK)
		railway = DashedLine(start = self.tunnel_center-self.tunnel_width/2*RIGHT-self.tunnel_height/2*UP, end = self.tunnel_center-(self.tunnel_width/2+3)*RIGHT-self.tunnel_height/2*UP, stroke_width = 4, stroke_color = GRAY)

		self.plot_range = [0, 10, 2]
		self.ax = Axes(x_length = self.tunnel_width, y_length = self.tunnel_height, x_range = self.plot_range, y_range = [-1, 1, 1], 
			axis_config = {"stroke_width": 1, "stroke_opacity": 1, "tip_width": 0.125, "tip_height": 0.125, "stroke_color": BLACK}).move_to(self.tunnel_center+0.05*LEFT)
		self.add(upper_tunnel, lower_tunnel, railway)


	def show_pressure(self, t):
		n_lines = 400
		line_pressure_group = VGroup()
		for i in range(n_lines):
			line_x_coord = self.plot_range[0] + self.plot_range[1] / (n_lines-1) * i
			pressure_pulse = get_pressure_pulse(t)
			line_alpha = (pressure_pulse(line_x_coord) + 1) / 2
			line_pressure = Line(start = self.ax.c2p(line_x_coord, -1), end = self.ax.c2p(line_x_coord, 1), stroke_color = BLACK, stroke_opacity = line_alpha, stroke_width = 1)
			# debug_dot = Dot(color = PURE_BLUE, radius = 0.01).move_to(self.ax.c2p(line_x_coord, pressure_pulse(line_x_coord)))
			line_pressure_group.add(line_pressure)
		return line_pressure_group

        


class TunnelPressure(Mobject):
	def __init__(self, tunnel_center = np.array([0, 0, 0]), tunnel_height = 1, tunnel_width = 5, **kwargs):
		super().__init__(**kwargs)
        
		self.tunnel_center = tunnel_center
		self.tunnel_height = tunnel_height
		self.tunnel_width = tunnel_width

		self.plot_range = [0, 10, 2]
		self.ax = Axes(x_length = self.tunnel_width, y_length = self.tunnel_height, x_range = self.plot_range, y_range = [-1, 1, 1], 
			axis_config = {"stroke_width": 1, "stroke_opacity": 1, "tip_width": 0.125, "tip_height": 0.125, "stroke_color": BLACK}).move_to(self.tunnel_center)
		self.add(self.ax)


	def show_pressure(self, t):
		# pressure_left_func = get_interval_pressure_left(t)
		# pressure_right_func = get_pressure_left(t)
		pressure_pulse = get_pressure_pulse(t)
		pressure_plot = self.ax.plot(pressure_pulse, x_range = [*self.plot_range[:2]], stroke_color = PURE_RED)
		return pressure_plot

In [344]:
%%manim -qh --fps 60 $video_scene


class tunnel_compression_Scene(Scene):
	def construct(self):
		self.camera.background_color = WHITE

		CVC = Text('CVC', font_size = 12, weight = BOLD, color = WHITE, font = 'Latin Modern Sans').align_on_border(RIGHT + DOWN, buff = 0.2)
		self.add(CVC)


		tunnel_center = np.array([1.5, 2, 0])
		tunnel_ax_center = np.array([1.5, 0.5, 0])

		tunnel_height = 0.75
		tunnel_width = 8.0

		t = 0

		tunnel = Tunnel(tunnel_center = tunnel_center, tunnel_height = tunnel_height, tunnel_width = tunnel_width)
		tunnel_pressure = tunnel.show_pressure(t)
		self.add(tunnel, tunnel_pressure)


		tunnel_ax = TunnelPressure(tunnel_center = tunnel_ax_center, tunnel_height = 0.75, tunnel_width = tunnel_width)
		tunnel_ax_plot = tunnel_ax.show_pressure(t)
		self.add(tunnel_ax, tunnel_ax_plot)

		# updates the pressure inside the tunnel
		def tunnel_pressure_updater(pressure):
			time = time_tracker.get_value()
			tunnel_pressure_new = tunnel.show_pressure(time)
			pressure.become(tunnel_pressure_new)
		

		# updates the plot inside the tunnel
		def tunnel_ax_plot_updater(plot):
			time = time_tracker.get_value()
			tunnel_plot_new = tunnel_ax.show_pressure(time)
			plot.become(tunnel_plot_new)


		time_tracker = ValueTracker(t)
		
		self.wait(1.5)
		tunnel_ax_plot.add_updater(tunnel_ax_plot_updater)
		tunnel_pressure.add_updater(tunnel_pressure_updater)
		self.play(time_tracker.animate.set_value(15), rate_func = linear, run_time = 15) 
		self.wait(3)

Manim Community v0.18.1

8.0 8.0
